In [1]:
BATCH_MODE = "true"

# --- Taille du rejeu (critere D2 de #24282 ; cadran de puissance de #26829) ---
# N_H_GROUPES : nombre de groupes H uniques (textes positifs distincts) retenus
#               par la stratification.
# REPS_PAR_H  : repetitions par groupe H -- 1 appariement par longueur +
#               (REPS_PAR_H - 1) en round-robin sur les non-humour restants.
#               Le ratio de classe du dispositif est donc 1 groupe H pour
#               REPS_PAR_H non-humour.
# 12 groupes H -> 12 + 60 = 72 unites group-level : premiere ligne du cadran de
# #26829 ou la sonde voit un effet de d >= 2.0 a 80 % de puissance (la taille
# precedente, 6 groupes / 36 unites, ne voyait que d >= 2.5).
# Le corpus porte 48 instances 'humour_reussi' de textes distincts : les 6
# groupes de la premiere version etaient un CHOIX de stratification, pas une
# limite de donnees. C'est ce que le rejeu (#26911 etape 3) mesure.
N_H_GROUPES = 12
REPS_PAR_H = 5


> **Statut épistémique** — **Sans verdict à ce jour** : aucune ligne de la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut épistémique sera porté par la matrice le cas échéant.

**Navigation** : [Index ICT-Series](README.md) | [>> ICT-36 (F-Lens, factored-geometry)](ICT-36-FLens-FactoredGeometry-Python.ipynb)


> **Acquis et parenté** : strate 5 (ICT-21 → ICT-25 prolongée par ICT-35 → ICT-39 — le LLM comme substrat). Étage 0 du programme HumorCausalProbe ([#5635](https://github.com/jsboige/CoursIA/issues/5635), [#8236](https://github.com/jsboige/CoursIA/issues/8236)) : la baseline de shortcuts lexicaux se mesure **avant** toute phase SAE, les phases suivantes restant aval.


## ICT-35 -- HumorCausalProbe-Pilot : substrat HLS, sans GPU, sans confusion explication/mecanisme

> **Serie ICT** (*Integrated Causal Trajectories*, EPIC #4588) -- strate 5 : *extraction GPU-free*.
> Issue : **#14035**. prerequis livre : banc humour dur **consolide dans GT-18c** (120 instances annotees, #12785 puis absorption #14032).
> Aval : #5635 (Gate 24 workspace), #8236 (multi-echelle), #5105 (Inoculation).

## Étage 0 -- baseline de shortcuts lexicaux (PAS Phase 1 SAE)

Ce notebook est l'**etage 0** d'un escalier qui monte vers un protocole d'humour falsifiable :

1. **Surface lexicale** (ce notebook, ICT-35-Pilot) -- 6 features lexicales deterministes sur 60 appariements ; verdict **borne** sur le sous-corpus.
2. **Embeddings / modele gele** (a venir) -- features continues sur la zone editee (sortie d'un encodeur fixe).
3. **Modele generatif avec contexte narratif** (a venir) -- discrimination sur la sortie textuelle complete.
4. **SAE / J-lens** (#14035 horizon) -- features sparse sur les activations internes ; le protocole Phase 1 (placeholder ci-dessous) ne consomme QUE l'etage 0.

A chaque etage, on mesure le **gain par niveau de taxonomie** avec intervalles de confiance (pas une moyenne globale). Le verdict de cet etage 0 ne **falsifie** ni l'humour lexical en general, ni un mecanisme interne : il **borne** la discrimination de ces 6 features sur ce corpus -- une mesure reproductible, pas un verdict fort.

## Pourquoi ce notebook existe

Le banc humour de la serie GameTheory (GT-28 / GT-28b) mesure les **sorties** d'un LLM classifieur. Il dit quelque chose sur la difficulte du probleme (F1 0,5-0,6 sur 5 classes), pas sur le **mecanisme** interne. Les surveys HLS placent l'humour parmi les semantiques de haut niveau qui exigent contexte, chaines causales et pragmatique -- et avertissent que les explications textuelles post-hoc sont rarement fideles au mecanisme reel.

Ce notebook pose l'**etage 0 lexical** du protocole de l'issue #14035 :

1. **60 appariements exploratoires** `humour_reussi` <-> `unfun` issus du CORPUS_DUR ; **12 groupes H uniques** reutilises 5 fois chacun, **60 negatifs uniques** apparies par longueur (+/-20 %) pour isoler la dimension humoristique de la dimension verbosite. C'est une mesure exploratoire sur un sous-ensemble stratifie, pas un test sur 60 paires minimales independantes.
2. **2 controles negatifs** : (a) **shuffle** des labels (memes textes, attribution aleatoire) ; (b) **surface-only** = blagues reussies amputees du recadrage (sanity check que la discrimination n'est pas triviale, **borne** a ces 6 transformations).
3. **Mesure de stabilite** : inter-prompts (3 reformulations par instance), inter-formes (ponctuelles, sociales, topical).
4. **Classifieur lexical** (regression logistique regularisee sur 6 features lexicales simples, **ajustee INTRA-fold** via `Pipeline` pour eviter la fuite de StandardScaler) : F1 hold-out pour discriminer `humour_reussi` vs `non-humour`.
5. **Verdict borne** : `FEATURE_CANDIDATE` (F1 > 0,65, donc les features lexicales capturent quelque chose au-dela de la surface) / `SURFACE_SEULE` (F1 <= 0,55) / `INCONCLUSIVE` (entre les deux). Le verdict porte sur la capacite des 6 features lexicales a **discriminer sur des positifs jamais vus** (StratifiedKFold n_splits=3 sur 72 unites group-level = 12 H + 60 N distincts), pas sur la memorisation des 12 positifs en CV stratifiee par instance.

## Ce que ce notebook ne fait PAS

- **Pas de GPU**. C'est l'etage 0 lexical, l'agent du worktree n'a pas de GPU.
- **Pas de SAE ni de transformer reel**. Les features lexicales servent de **baseline de shortcuts** mesurable hors-ligne ; un futur etage (gated par #5635 GPU Gate 24) consommera ce protocole en remplacant features lexicales -> features SAE.
- **Pas de confusion explication/mecanisme**. Aucune explication generee par LLM n'est traitee comme preuve de mecanisme interne. C'est la clause 6 de l'acceptance #14035.
- **Pas d'oracle humain**. Les labels `humour_reussi` ne sont pas des jugements humains : ils viennent de la chaine de qualification (#14032 pour la methode, #26911 etape 2 pour la mesure de l'accord). L'accord de cette chaine avec une reference humaine echantillonnee est **mesure** dans l'artefact commite (#26924) : **kappa = 0.863** (accord corrige du hasard ; accord observe 0.917, accord attendu 0.390), soit un **desaccord brut de 5 items sur 60 (8,3 %)** -- sur les cinq classes, pas sur la seule classe positive. Ce bruit d'annotation borne le signal que cette sonde peut chercher : un F1 mesure ici se compare au **null intra-fold**, jamais a un oracle humain. Le kappa ne fixe **aucun plafond de F1** : la matrice de confusion et la distinction des quantites sont publiees en Lecture 5 bis. La limite est reprise dans les limites du verdict (cellule de verdict, point 5) et dans le « A retenir ».
- **Pas de generalisation a d'autres corpus**. Le verdict porte strictement sur les 60 appariements du sous-ensemble stratifie du banc dur ; il est **INCONCLUSIVE borne**, pas une falsification de l'humour lexical en general.

## La taille demandee, et ce que le corpus permet (critere D2, #26829)

Le cadran de puissance ajoute par #26829 (le carnet lui) montre que, a 36 unites group-level, cette sonde ne voit a 80 % de puissance qu'un effet de taille `d >= 2.5` : sa borne etait large, et son `INCONCLUSIVE` ne pouvait rien exclure de moyen. Le rejeu de ce carnet porte donc le sous-ensemble stratifie a **12 groupes H** (72 unites group-level), la taille ou le cadran montre qu'un effet `d = 2` devient visible a 80 % de puissance.

Ce que le corpus **ne permet pas** : la ligne 24 groupes H / 144 unites du cadran exigerait **120 negatifs uniques** (5 N par groupe H), et le corpus n'en compte que **72**. La taille maximale atteignable avec le corpus actuel est donc **14 groupes H (84 unites group-level)** ; la ligne 24/144 reste hors d'atteinte sans un corpus etendu (voie #14033). Ce carnet mesure une taille **atteignable** -- les 12 groupes H du rejeu -- **en deca** de ce plafond de 14 : les deux chiffres ne se confondent pas, 12 est la grille retenue, 14 le plafond du corpus.

## Sequencage developemental (juste pour memoire, sans age invente)

Ce notebook est volontairement **circonscrit a l'etage 0 lexical**. Les etages suivants dependent d'un substrat que CE notebook ne construit pas :

- **#14032** consolide la **methode de labellisation** du GT-28 / GT-28b (heuristique Argumentum + or humain echantillonne). Sans substrat propre, aucun etage superieur est lancable.
- **#14033** livre le **corpus apparie** avec 3 labels independants par instance (structure / cible pragmatique / fonction encyclopedique) et la **taxonomie croissante des capacites** que les etages superieurs discrimineront. La taxonomie est **sourcee** (Attardo/Raskin general theory of verbal humor + Mihalcea/Pulido humor stylometry), pas inventee dans cette PR. **Ce notebook n est PAS adosse a #14033** : il est exploratoire et precede l arrivee du substrat taxonomique. L acceptance #14035 " >= 30 paires minimales" est **decochee** pour ce sous-grain (verdict reformule : 60 appariements exploratoires sur 12 groupes H uniques, pas 60 paires minimales).
- **#14035** (cible) consomme le substrat de #14033 pour passer d'embeddings + SAE a un protocole complet. L'etage 0 lexical (ce notebook) est **strictement anterieur** a #14033 et ne court-circuite pas la taxonomie.

Taxonomie croissante (capacites discriminant l'humour, sourcee -- pas d'ordre d'age invente) :

1. **Jeu sonore / repetition + incongruite perceptible** (calembours, repetitions structurelles).
2. **Incongruite / resolution semantique simple** (renversement attendu a porte close).
3. **Recit, attente et chute** (structures narratives classiques).
4. **Perspective d'autrui, empathie, normes sociales** (humour sur le rapport a l'autre).
5. **Implicite pragmatique, ironie, double audience** (le non-dit, le sous-entendu).
6. **References culturelles / topicales, contexte long** (encyclopedie partagee).

L'etage 0 ne discrimine **que** la couche 1 (jeu sonore / repetition + incongruite perceptible) **sur les 12 groupes H uniques du sous-corpus** -- les 5 autres capacites demandent l'etage embeddings ou contexte narratif. La discrimination est mesuree par StratifiedKFold(n_splits=3) sur 72 unites group-level (1 par groupe H = moyenne des 5 repetitions + 60 N singletons ; un meme texte positif source n apparait jamais en train et test d'un meme fold), pas par memorisation des 12 textes en CV stratifiee.

## Sources

- `MyIA.AI.Notebooks/GameTheory/GameTheory-18c-Humour-Banc-Python.ipynb` : CORPUS_DUR 120 instances (cell[33], section echelle consolidee #14032), labellisation manuelle par heuristique Argumentum (cf note methodo de l'acceptance).
- `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\Computational Humor\` : sources HLS (Attardo/Raskin GTVH + Mihalcea/Pulido humor stylometry), **prevues comme reference** pour la taxonomie -- la mise en archive effective est a faire dans le scope de #14033, pas de ce notebook.


## Le protocole de mesure : 72 unites group-level

L'unite de mesure n'est pas l'instance mais le **groupe**. Chaque texte `humour_reussi` apparait 5 fois dans les 60 paires (une repetition par appariement, 12 groupes uniques) : mesurer au niveau instance laisserait le classifieur reconnaitre le texte source plutot que la classe. Le jeu de mesure reduit donc chaque groupe H a **une seule unite = moyenne des 5 repetitions**, et prend les 60 negatifs uniques comme singletons : **72 unites** (12 H + 60 N).

Trois choix structurent la mesure :

1. **Split** : `StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)` sur `y_group` -- 24 unites par fold (4 H + 20 N en test). Les repetitions etant deja reduites a une moyenne par H, un meme texte positif source n'apparait jamais a la fois en train et en test d'un meme fold.
2. **Controle de permutation** : pour chaque permutation, les labels des 72 unites sont permutes au niveau global (distribution 12H/60N preservee), le pipeline est **refit intra-fold sur les labels permutes**, et on ne retient qu'**une seule moyenne CV par permutation**. La distribution nulle compte 100 moyennes, et le p-value est **fini** : `(1 + k) / (N_PERM + 1)` ou k = nombre de moyennes nulles >= F1 reel (Phipps & Phipps / North et al., borne superieure du vrai p sous H0). Permuter les labels de test seul, a modele fixe, n'en serait pas un : le modele doit etre re-entraine sur les labels permutes.
3. **Regle de verdict conservatrice** : `INCONCLUSIVE borne` des que le p-value fini >= 0,05 OU que l'IC95 du `delta_proba` OOF traverse 0. Le verdict `SURFACE_SEULE` n'est jamais declare sur ce N : avec 12 items, un IC95 qui traverse 0 est trivialement attendu ; `SURFACE_SEULE` demanderait un IC95 entierement au-dessus de 0 sur un N suffisant.

Ce protocole tient son verdict **structurellement** sur ce sous-corpus : 72 unites, p fini, IC95, regle de decision conservatrice.

In [2]:
# -*- coding: utf-8 -*-
# Setup. Pas de service externe ; pas de GPU. CPU-only, sklearn, json, random.
import json
import random
import re
import string
from collections import Counter, defaultdict
from pathlib import Path
import pickle
import time

import numpy as np

# sklearn : classifieur lexical baseline. CPU only, deterministe.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

SEED = 13306
random.seed(SEED)
np.random.seed(SEED)

GT28B = Path("MyIA.AI.Notebooks/GameTheory/GameTheory-18c-Humour-Banc-Python.ipynb")  # consolide #14032
assert GT28B.exists(), f"banc humour 18c manquant : {GT28B.resolve()}"
print(f"[setup] banc humour 18c (consolide #14032) present : {GT28B}")
print(f"[setup] seed = {SEED}")
print(f"[setup] sklearn LogisticRegression baseline (CPU)")


[setup] banc humour 18c (consolide #14032) present : MyIA.AI.Notebooks\GameTheory\GameTheory-18c-Humour-Banc-Python.ipynb
[setup] seed = 13306
[setup] sklearn LogisticRegression baseline (CPU)


In [3]:
# Chargement du corpus humour depuis l'organe (extraction #20316, T7') :
# humour_banc/ = fichiers de données committés + chargeur canonique. La
# couche ict délègue à l'organe — plus d'exécution positionnelle des cellules
# du banc 18c (fragile à la renumérotation, exigeait la clé OpenRouter et un
# fetch réseau).
import sys

# Le pilote s'execute depuis la racine du depot (cf. GT28B, cellule 4) :
# le package ict est ancre par chemin relatif explicite.
ICT_ROOT = Path("MyIA.AI.Notebooks/IIT/ICT-Series")
if str(ICT_ROOT) not in sys.path:
    sys.path.insert(0, str(ICT_ROOT))

from ict.humor_pairs import load_corpus_dur, validate_corpus

CORPUS_DUR = load_corpus_dur()
validate_corpus(CORPUS_DUR)
print(f"[extract] CORPUS_DUR (organe humour_banc) : {len(CORPUS_DUR)} instances")
print(f"[extract] distribution par label : {dict(Counter(i['label'] for i in CORPUS_DUR))}")
print(f"[extract] distribution par source : {dict(Counter(i['source'] for i in CORPUS_DUR))}")

# Compatibilité RNG : l'ancienne reproduction exécutait la cellule 33 du banc
# (random.seed(42) puis sample de 60 instances sur 167), ce qui fixait l'état
# du RNG consommé ensuite par la stratification (cellule 7 ci-dessous). On
# rejoue exactement cette trajectoire pour que H_REUSSI — donc toutes les
# lectures committées — soient identiques.
random.seed(42)
random.sample(range(167), 60)

[extract] CORPUS_DUR (organe humour_banc) : 120 instances
[extract] distribution par label : {'rire_sans_recadrage': 14, 'recadrage_sans_rire': 13, 'rien': 38, 'humour_reussi': 48, 'offensif_compris_non_partage': 7}
[extract] distribution par source : {'ArgumentumGames/Argumentum': 60, 'blague-manuelle': 30, 'declaration-factuelle': 20, 'edge-case-curated': 10}


[163,
 28,
 6,
 70,
 62,
 57,
 35,
 26,
 139,
 22,
 151,
 108,
 8,
 7,
 23,
 55,
 59,
 129,
 164,
 143,
 50,
 158,
 107,
 56,
 114,
 71,
 1,
 40,
 155,
 87,
 141,
 39,
 156,
 86,
 159,
 152,
 97,
 24,
 91,
 88,
 77,
 33,
 103,
 5,
 93,
 58,
 68,
 15,
 118,
 48,
 10,
 166,
 37,
 106,
 80,
 79,
 110,
 46,
 73,
 149]

## Lecture 1 -- Le sous-corpus : 120 instances, un label a 48

La sortie du chargement (organe `humour_banc/`, extraction #20316) fixe le terrain : CORPUS_DUR = 120 instances, 5 categories ({'rire_sans_recadrage': 14, 'recadrage_sans_rire': 13, 'rien': 38, 'humour_reussi': 48, 'offensif_compris_non_partage': 7}) et 4 sources ({'ArgumentumGames/Argumentum': 60, 'blague-manuelle': 30, 'declaration-factuelle': 20, 'edge-case-curated': 10}).

Le label positif `humour_reussi` est la categorie la plus fournie du corpus brut (48 sur 120, soit 40 %) : la difficulte ne vient donc **pas** de sa rarete. La stratification en retient **12** groupes H (cellule suivante) -- la taille **retenue par ce protocole**, et non le maximum que le corpus permettrait : avec les 72 negatifs disponibles et le ratio de 5 N par groupe H, le corpus pourrait aller jusqu'a **14 groupes H (84 unites group-level)**. Les 12 groupes sont donc un choix de protocole, a une marche du plafond du corpus -- c'est le 14 qui est le plafond, pas le 12.

C'est la lecture a retenir ici : la taille du sous-corpus etait un **choix de protocole**, pas une limite de donnees. Le corpus offrait 48 textes positifs distincts et 72 negatifs ; la premiere version du carnet n'en retenait que 6 groupes H, sous un regime ou le cadran de #26829 montre que la sonde ne pouvait voir qu'un effet de taille `d >= 2.5`.

In [4]:
# Stratification : N_H_GROUPES textes positifs uniques (label 'humour_reussi'),
# tous les non-humour disponibles.
# Le corpus porte 48 instances 'humour_reussi' **de textes distincts** (30
# blagues manuelles + 18 Argumentum) : les 6 groupes retenus par la premiere
# version du carnet etaient un **choix** de stratification, pas une limite de
# donnees. Le rejeu (#26911 etape 3) porte ce nombre a 12 groupes H -- la
# taille que le cadran de #26829 exige pour voir un effet de d >= 2.0 a 80 %
# de puissance.
# Le ratio 1:5 maximise la variete des non-humour pour la diversite des
# negatifs.
# Echantillonnage **par groupe H** : l'unite de mesure est le **groupe**
# (une seule unite par H = moyenne des repetitions), construite au build
# group-level.
# Le split utilise StratifiedKFold(n_splits=3) sur y_group : coherent avec une
# unite-de-groupe ou les repetitions ont deja ete reduites a une moyenne par H.
by_label = defaultdict(list)
for inst in CORPUS_DUR:
    by_label[inst['label']].append(inst)

# humour_reussi : N_H_GROUPES instances (textes uniques).
random.shuffle(by_label['humour_reussi'])
H_REUSSI = by_label['humour_reussi'][:N_H_GROUPES]

# non-humour : TOUTES les instances des 4 autres labels.
# Scope exploratoire : on consomme la totalite des non-humour pour maximiser la
# diversite des negatifs sur les appariements exploratoires (REPS_PAR_H
# repetitions par groupe H). Sub-grain limite volontairement avant tout substrat
# plus expressif (cf #14033 substrat taxonomique croissant, etape d apres).
NON_HUMOUR = []
for label in ['rire_sans_recadrage', 'recadrage_sans_rire', 'rien', 'offensif_compris_non_partage']:
    NON_HUMOUR.extend(by_label[label])

print(f"[stratify] humour_reussi = {len(H_REUSSI)} instances")
print(f"[stratify] non-humour    = {len(NON_HUMOUR)} instances (toutes disponibles)")
print(f"[stratify] ratio H:N     = 1:{len(NON_HUMOUR)//len(H_REUSSI)}")


[stratify] humour_reussi = 12 instances
[stratify] non-humour    = 72 instances (toutes disponibles)
[stratify] ratio H:N     = 1:6


In [5]:
# Appariement par longueur (token count approximatif, +/-20 %).
# On definit la longueur d'une instance par le nombre de mots de son texte.
def text_length(inst):
    return len(inst['texte'].split())

H_LEN = sorted([(text_length(i), i) for i in H_REUSSI], key=lambda x: x[0])
N_LEN = sorted([(text_length(i), i) for i in NON_HUMOUR], key=lambda x: x[0])

print("[match] longueurs des", len(H_REUSSI), "humour_reussi :", [l for l, _ in H_LEN])
print(f"[match] longueurs non-humour (n={len(N_LEN)}) : min={min(l for l,_ in N_LEN)} max={max(l for l,_ in N_LEN)}")

# Pour chacun des groupes H, on cherche le non-humour le plus proche en longueur.
PAIRES = []  # (humour, non-humour, delta_len)
used = set()
for hl, hi in H_LEN:
    candidates = [(abs(nl - hl), ni) for nl, ni in N_LEN if id(ni) not in used]
    candidates.sort(key=lambda x: x[0])
    if not candidates:
        break
    delta, ni = candidates[0]
    used.add(id(ni))
    PAIRES.append((hi, ni, delta))

print(f"[match] {len(PAIRES)} paires appariees par longueur (+/-20%)")
for i, (h, n, d) in enumerate(PAIRES):
    print(f"  P{i+1:02d} humour={text_length(h):3d} mots  <->  unfun={text_length(n):3d} mots  delta={d:2d}")

# Scope exploratoire honnete : pas de moitie, pas de Phase 1. PAIRES_EXPL etendu par
# round-robin sur les non-humour restants (jusqu'a epuisement).
print(f"[match] appariements = {len(PAIRES)} par longueur + {REPS_PAR_H - 1} par groupe H "
      f"en round-robin sur les non-humour restants")
print(f"        ({REPS_PAR_H - 1} supplementaires par groupe H, jusqu'a epuisement des non-humour).")

PAIRES_EXPL = list(PAIRES)
remaining = [n for _, n, _ in [(text_length(n), n, None) for n in NON_HUMOUR] if id(n) not in {id(x) for _, x, _ in PAIRES}]
for h_idx, (hl, hi) in enumerate(H_LEN):
    for k in range(REPS_PAR_H - 1):
        if not remaining:
            break
        ni = remaining.pop(0)
        PAIRES_EXPL.append((hi, ni, abs(text_length(ni) - text_length(hi))))
# Un negatif distinct par appariement (les paires par longueur marquent `used`,
# le round-robin puise dans `remaining`) : le nombre de negatifs uniques est donc
# exactement le nombre d'appariements.
print(f"[match] PAIRES_EXPL total = {len(PAIRES_EXPL)} appariements exploratoires "
      f"({len(H_REUSSI)} groupes H x {REPS_PAR_H} repetitions, "
      f"{len(PAIRES_EXPL)} negatifs uniques -- un negatif distinct par appariement)")

# Group key pour le split group-level : identifie le texte positif source
# (une valeur unique par groupe H). Les groupes H sont distribues dans les 3 folds
# par StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED) sur y_group ;
# les N sont leurs propres singletons distribues par stratification de la classe
# positive.
GROUP_KEY_H = [p[0]['texte'] for p in PAIRES_EXPL]   # len(PAIRES_EXPL) cles, len(H_REUSSI) uniques
n_groupes_uniques = len(set(GROUP_KEY_H))
print(f"[match] Groupes uniques (textes positifs sources) = {n_groupes_uniques} (max {len(PAIRES_EXPL)//n_groupes_uniques} repetitions par groupe)")


[match] longueurs des 12 humour_reussi : [10, 10, 14, 14, 16, 19, 20, 20, 22, 22, 23, 23]
[match] longueurs non-humour (n=72) : min=5 max=29
[match] 12 paires appariees par longueur (+/-20%)
  P01 humour= 10 mots  <->  unfun= 10 mots  delta= 0
  P02 humour= 10 mots  <->  unfun= 10 mots  delta= 0
  P03 humour= 14 mots  <->  unfun= 14 mots  delta= 0
  P04 humour= 14 mots  <->  unfun= 14 mots  delta= 0
  P05 humour= 16 mots  <->  unfun= 16 mots  delta= 0
  P06 humour= 19 mots  <->  unfun= 19 mots  delta= 0
  P07 humour= 20 mots  <->  unfun= 20 mots  delta= 0
  P08 humour= 20 mots  <->  unfun= 19 mots  delta= 1
  P09 humour= 22 mots  <->  unfun= 22 mots  delta= 0
  P10 humour= 22 mots  <->  unfun= 22 mots  delta= 0
  P11 humour= 23 mots  <->  unfun= 23 mots  delta= 0
  P12 humour= 23 mots  <->  unfun= 23 mots  delta= 0
[match] appariements = 12 par longueur + 4 par groupe H en round-robin sur les non-humour restants
        (4 supplementaires par groupe H, jusqu'a epuisement des non-humour

## Lecture 2 -- Appariement par longueur : 12 + 48, deltas nuls

La cellule d'appariement produit deux choses. D'abord 12 paires par longueur, **quasi exactement egales** (delta = 0 pour 11 d'entre elles ; une seule derive d'un mot : un humour de 20 mots apparie a un non-humour de 19 -- longueurs cote humour : [10, 10, 14, 14, 16, 19, 20, 20, 22, 22, 23, 23]). Ensuite 48 paires supplementaires en round-robin sur les non-humour restants (4 par groupe H), jusqu'a epuisement — total : 60 appariements exploratoires, 60 negatifs uniques (un negatif distinct par appariement), 12 groupes H sources.

La lecture a retenir : l'appariement +/-20 % annonce dans le frontmatter est en pratique **exact ou quasi exact** sur les 12 paires initiales — la longueur ne confond pas la discrimination, c'est une variable controlee au mot pres. Et le design 12 H x 5 repetitions est entierement determine par la taille du sous-ensemble stratifie : 60 appariements exploratoires, pas 60 paires minimales independantes. C'est la limite structurelle citee en ouverture, chiffree ici — et elle est la meme qu'a 6 groupes H, seulement portee a une echelle double.


In [6]:
# Features lexicales. 6 dimensions, deterministes, sans dependance externe.
# Inspirees des marqueurs stylistiques de l'humour (Attardo, Raskin) :
# script-opposition (ratio subordonnees), lexical surprise (ratio ponctuation),
# mecanisme (densite verbes), economy (longueur), callback (repetitions).
PUNCT = set(string.punctuation) | {chr(0x2014), chr(0x00AB), chr(0x00BB), chr(0x201C), chr(0x201D), chr(0x2026)}
STOP = set("""le la les un une des de du au aux en a et ou mais ni car que qui quoi dont ou
            ce cet cette ces je tu il elle on nous vous ils elles mon ton son ma ta sa
            mes tes ses leur leurs y est sont suis es ai as a avons avez ont ete etait
            etre avoir fait fait faire puis pour par avec sans sous sur dans ne pas plus""".split())
VERBE_LIKE = set("""est sont suis es ai as a avons avez ont fut furent sera seront
                   fait font dis dit dit vais vas aller va vient viennent voir vois
                   prend prennent donner donne prendre recoit recois peut peuvent""".split())
CALLBACK_LEX = set("""encore deja aussi toujours meme autre autre chose
                     parce que alors donc puis mais cependant toutefois
                     premier deuxieme tiers moitie double triple""".split())

def features(inst):
    text = inst['texte']
    tokens = re.findall(r"[A-Za-zÀ-ÿĀ-ſ']+", text)
    n_tok = max(1, len(tokens))
    lower = [t.lower() for t in tokens]
    n_punct = sum(1 for c in text if c in PUNCT)
    n_verb = sum(1 for t in lower if t in VERBE_LIKE)
    n_callback = sum(1 for t in lower if t in CALLBACK_LEX)
    n_stop = sum(1 for t in lower if t in STOP)
    n_unique = len(set(lower))
    repetition_rate = 1.0 - (n_unique / n_tok) if n_tok else 0.0
    return {
        'n_tokens':      float(n_tok),
        'n_punct':       float(n_punct),
        'n_verb':        float(n_verb),
        'n_callback':    float(n_callback),
        'n_stop':        float(n_stop),
        'repetition':    float(repetition_rate),
    }

def vec(inst):
    f = features(inst)
    return [f['n_tokens'], f['n_punct'], f['n_verb'],
            f['n_callback'], f['n_stop'], f['repetition']]

def label_y(inst):
    return 1 if inst['label'] == 'humour_reussi' else 0

# Affiche les features sur le premier exemple
sample = H_REUSSI[0]
f = features(sample)
print(f"[feat] exemple humour_reussi (id={sample['id']})")
for k, v in f.items():
    print(f"  {k:12s} = {v:.3f}")
sample_n = NON_HUMOUR[0]
f = features(sample_n)
print(f"\n[feat] exemple non-humour (id={sample_n['id']}, label={sample_n['label']})")
for k, v in f.items():
    print(f"  {k:12s} = {v:.3f}")


[feat] exemple humour_reussi (id=joke-p27)
  n_tokens     = 10.000
  n_punct      = 2.000
  n_verb       = 0.000
  n_callback   = 1.000
  n_stop       = 4.000
  repetition   = 0.000

[feat] exemple non-humour (id=arg-7.3.2, label=rire_sans_recadrage)
  n_tokens     = 25.000
  n_punct      = 11.000
  n_verb       = 1.000
  n_callback   = 1.000
  n_stop       = 8.000
  repetition   = 0.120


In [7]:
# Build group-level : **une unite par groupe H** (moyenne des repetitions) +
# une unite par N singleton. Un shuffle au niveau des repetitions serait
# tautologique -- toutes les cles H sont a y=1 : permuter les cles entre elles
# laisse y inchange. StratifiedKFold sur y_group (3 folds, seed=SEED) ;
# permutations intra-fold sur y_group (preservation ~1/5 de H par bloc test) ;
# surface rappel + delta OOF proba positive. StandardScaler integre au Pipeline
# intra-fold : la permutation de y_group intra-fold est un null control
# informatif sur les unites group-level.
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# Reconstruire X/y sur les instances appariees pour les utiliser dans le group-level build.
paire_H = [p[0] for p in PAIRES_EXPL]
paire_N = [p[1] for p in PAIRES_EXPL]
instances_paires = paire_H + paire_N
X = np.array([vec(i) for i in instances_paires])
y = np.array([label_y(i) for i in instances_paires])
N_KEY = [p[1]['texte'] for p in PAIRES_EXPL]
print(f"[Xy] X.shape = {X.shape}, y.shape = {y.shape}, N_KEY len = {len(N_KEY)}")

# H_KEY_TO_POSITIONS : positions des repetitions de chaque texte H dans PAIRES_EXPL
# (necessaire au build group-level ci-dessous).
H_KEY_TO_POSITIONS = {}
for pos, key in enumerate(GROUP_KEY_H):
    H_KEY_TO_POSITIONS.setdefault(key, []).append(pos)
print(f"[Xy] H_KEY_TO_POSITIONS : {len(H_KEY_TO_POSITIONS)} groupes H uniques")

# Moyenne par groupe : une unite par H (moyenne des repetitions) + une unite par
# N distinct. Meme longueur X = 6 features.
X_group = []
y_group = []
group_keys = []   # len(group_keys) cles uniques (groupes H + N distincts)
for h_key, positions in H_KEY_TO_POSITIONS.items():
    vecs = X[positions]   # vecteurs des repetitions du groupe H
    X_group.append(vecs.mean(axis=0))
    y_group.append(int(y[positions[0]]))   # 1 pour H
    group_keys.append(h_key)
# Les N sont les singletons qui suivent les H dans instances_paires : leur index
# dans X est decale du nombre de paires H (pas du nombre de paires N).
for n_key, n_pos in zip(N_KEY, range(len(paire_H), len(paire_H) + len(paire_N))):
    X_group.append(X[n_pos])   # N deja singletons
    y_group.append(int(y[n_pos]))
    group_keys.append(n_key)
X_group = np.array(X_group)
y_group = np.array(y_group)
print(f"[Xy] X_group.shape = {X_group.shape}  y_group.shape = {y_group.shape}")
print(f"[Xy] y_group=1 (H) = {int(y_group.sum())}  y_group=0 = {int((1-y_group).sum())}")
print(f"[Xy] ratio classe positive = {y_group.mean():.3f}")
print(f"[Xy] group_keys uniques = {len(set(group_keys))} ({len(group_keys)} attendus)")

# StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED) sur y_group.
# Chaque fold : 1/3 des groupes H en test, le reste des N ; la stratification
# preservant le ratio de classe d'origine dans chaque bloc test.
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
SPLITS = list(skf.split(X_group, y_group))
for fold_idx, (tr, te) in enumerate(SPLITS):
    n_h_te = int(y_group[te].sum())
    n_n_te = int((1 - y_group[te]).sum())
    print(f"[Xy] fold {fold_idx}  n_train={len(tr)}  n_test={len(te)}  H_test={n_h_te}  N_test={n_n_te}")

print(f"[Xy] SPLITS materialisees : {len(SPLITS)} folds (3-fold SKF, memes seeds entre baseline/null/surface)")
print(f"[Xy] StandardScaler integre au Pipeline, refit intra-fold")


[Xy] X.shape = (120, 6), y.shape = (120,), N_KEY len = 60
[Xy] H_KEY_TO_POSITIONS : 12 groupes H uniques
[Xy] X_group.shape = (72, 6)  y_group.shape = (72,)
[Xy] y_group=1 (H) = 12  y_group=0 = 60
[Xy] ratio classe positive = 0.167
[Xy] group_keys uniques = 72 (72 attendus)


[Xy] fold 0  n_train=48  n_test=24  H_test=4  N_test=20
[Xy] fold 1  n_train=48  n_test=24  H_test=4  N_test=20
[Xy] fold 2  n_train=48  n_test=24  H_test=4  N_test=20
[Xy] SPLITS materialisees : 3 folds (3-fold SKF, memes seeds entre baseline/null/surface)
[Xy] StandardScaler integre au Pipeline, refit intra-fold


In [8]:
# Baseline 1 -- classifieur lexical GROUP-LEVEL (72 unites : 12 H + 60 N).
# StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED) sur y_group --
# 3 folds, 24 unites / fold, stratification 4H / 20N par fold test.
# StandardScaler integre au Pipeline, refit intra-fold.
from sklearn.metrics import f1_score

folds = []
y_pred_all = np.zeros_like(y_group)
y_proba_all = np.zeros((len(y_group), 2))
for fold_idx, (train_idx, test_idx) in enumerate(SPLITS):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, class_weight='balanced')),
    ])
    pipe.fit(X_group[train_idx], y_group[train_idx])
    y_pred = pipe.predict(X_group[test_idx])
    y_proba = pipe.predict_proba(X_group[test_idx])
    y_pred_all[test_idx] = y_pred
    y_proba_all[test_idx] = y_proba
    f1 = f1_score(y_group[test_idx], y_pred, zero_division=0)
    folds.append({'fold': fold_idx, 'f1': f1, 'n_train': len(train_idx), 'n_test': len(test_idx)})
    print(f"[cv] fold {fold_idx}  n_train={len(train_idx)}  n_test={len(test_idx)}  F1={f1:.3f}")

f1s = [f['f1'] for f in folds]
f1_real = np.mean(f1s)
f1_real_std = np.std(f1s)
print(f"[cv] F1 group-level baseline = {f1_real:.3f} +/- {f1_real_std:.3f}")
print(f"[cv] baseline random (ratio classe) F1 = {2 * y_group.mean() / (1 + y_group.mean()):.3f}")

# Probabilite OOF moyenne sur les groupes H (utile pour surface delta OOF plus tard).
h_mask = y_group == 1
p_pos_h_oof = y_proba_all[:, 1][h_mask]
print(f"[cv] P(y=1|x) OOF moyen sur H = {p_pos_h_oof.mean():.3f} +/- {p_pos_h_oof.std():.3f}")
print(f"[cv] P(y=1|x) OOF par H unique : {[f'{p:.3f}' for p in p_pos_h_oof]}")


[cv] fold 0  n_train=48  n_test=24  F1=0.118
[cv] fold 1  n_train=48  n_test=24  F1=0.308


[cv] fold 2  n_train=48  n_test=24  F1=0.500
[cv] F1 group-level baseline = 0.308 +/- 0.156
[cv] baseline random (ratio classe) F1 = 0.286
[cv] P(y=1|x) OOF moyen sur H = 0.477 +/- 0.225
[cv] P(y=1|x) OOF par H unique : ['0.785', '0.421', '0.120', '0.856', '0.520', '0.393', '0.336', '0.660', '0.157', '0.410', '0.354', '0.708']


## Lecture 3 -- F1 0.308 : a peine au-dessus du ratio de classe

La CV group-level (3 folds, 4 H en test par fold, 24 unites par fold) donne un F1 de `0.308 +/- 0.156` — folds 0.118, 0.308, 0.500 — pour une baseline aleatoire au ratio de classe de `0.286`. Le classifieur lexical est **a peine au-dessus de la baseline random** : sur 72 unites (12 H + 60 N), presque rien ne permet de deviner qui est H.

Deuxieme lecture, plus informative : la probabilite positive OOF moyenne sur les 12 H est `0.477 +/- 0.225`, et la liste par H est dispersee : ['0.785', '0.421', '0.120', '0.856', '0.520', '0.393', '0.336', '0.660', '0.157', '0.410', '0.354', '0.708'] — de 0.120 a 0.856. Aucun texte n'est confiant, aucun n'est nettement ecarte : le modele n'a appris ni une regle de classe, ni une idiosyncrasie unique. C'est le comportement attendu quand le signal reel est au-dela du lexique : le verdict INCONCLUSIVE porte au moins un diagnostic de dispersion.

Ce que le passage de 6 a 12 groupes H change : le F1 reel remonte de `0.217` (36 unites, 6 H) a `0.308` (72 unites, 12 H). Il ne faut pas y lire un progres du signal — la Lecture 4 montre que cette valeur reste **compatible avec le null**. Ce que la taille change, c'est la **capacite a exclure** : c'est exactement ce que le cadran de #26829 mesure, et c'est ce qui justifie le rejeu.


In [9]:
# Controle 1 -- PERMUTATION intra-fold de y_group.
#
# Piege evite : permuter seulement les labels de test, a modele fixe, n'est PAS un
# test de permutation -- le modele doit etre re-entraine sur les labels permutes.
# Sinon la proportion obtenue melange des scores pseudo-independants (3 folds par
# permutation) et n'est pas un p-value de permutation CV.
#
# Procedure (1 moyenne CV par permutation, p-value finie) :
#   pour chaque i in range(N_PERM) :
#     - Construire y_perm de longueur n_total = permutation des unites au niveau
#       GLOBAL, en preservant la distribution globale (meme ratio que
#       y_group d'origine).
#     - Pour chaque fold j in SPLITS :
#         - Refit pipe sur (X_group[train_idx], y_perm[train_idx]).
#         - Predire X_group[test_idx], F1 vs y_perm[test_idx].
#         - Stocker le F1 du fold j.
#     - Stocker UNE moyenne CV = mean([F1_0, F1_1, F1_2]) au perm i.
#   Distribution nulle = liste de N_PERM moyennes CV (100 valeurs, pas 300).
#   p_value = (1 + count(null >= reel)) / (N_PERM + 1)   # correction finie.
#
rng_perm = np.random.default_rng(SEED + 7)   # seed distinct pour reproductibilite
N_PERM = 100

n_h = int(y_group.sum())             # nombre global de groupes H
n_total = len(y_group)               # nombre d'unites group-level

perm_means = []   # 1 moyenne CV par permutation
for perm_i in range(N_PERM):
    # Permute les indices et assigne 1 aux n_h premiers (preservation distribution globale).
    perm = rng_perm.permutation(n_total)
    y_perm = np.zeros(n_total, dtype=int)
    y_perm[perm[:n_h]] = 1
    y_perm[perm[n_h:]] = 0
    # Meme SPLITS, memes seeds -- la SEULE difference vs baseline = les labels de y_perm.
    fold_f1s = []
    for train_idx, test_idx in SPLITS:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, class_weight='balanced')),
        ])
        pipe.fit(X_group[train_idx], y_perm[train_idx])
        y_pred = pipe.predict(X_group[test_idx])
        fold_f1s.append(f1_score(y_perm[test_idx], y_pred, zero_division=0))
    perm_means.append(float(np.mean(fold_f1s)))

perm_means = np.array(perm_means)
n_above_or_equal = int((perm_means >= f1_real).sum())
# p-value fini : Phipps & Phipps / North et al. -- (k+1)/(N+1) tient la borne
# superieure du vrai p-value quand le test est exact sous l'hypothese nulle.
p_value_perm = (1 + n_above_or_equal) / (N_PERM + 1)
perm_mean = float(perm_means.mean())
perm_std = float(perm_means.std())
perm_ic95_low = float(np.percentile(perm_means, 2.5))
perm_ic95_high = float(np.percentile(perm_means, 97.5))
delta_real_minus_perm = f1_real - perm_mean

print(f"[perm] distribution nulle (N_PERM={N_PERM} moyennes CV) :")
print(f"  mean = {perm_mean:.3f}  std = {perm_std:.3f}")
print(f"  IC95 = [{perm_ic95_low:.3f}, {perm_ic95_high:.3f}]")
print(f"  null >= reel = {n_above_or_equal}/{N_PERM}")
print(f"  p_value fini (k+1)/(N+1) = (1+{n_above_or_equal})/{N_PERM+1} = {p_value_perm:.4f}")
print(f"  F1 reel = {f1_real:.3f}, delta (reel - perm_mean) = {delta_real_minus_perm:+.3f}")
if p_value_perm < 0.05:
    print("[perm] VERDICT PERMUTATION : signal significatif (p < 0.05).")
else:
    print("[perm] VERDICT PERMUTATION : signal NON significatif (p >= 0.05) -- "
          "la discrimination observee est compatible avec le null intra-fold.")


[perm] distribution nulle (N_PERM=100 moyennes CV) :
  mean = 0.226  std = 0.092
  IC95 = [0.046, 0.410]
  null >= reel = 16/100
  p_value fini (k+1)/(N+1) = (1+16)/101 = 0.1683
  F1 reel = 0.308, delta (reel - perm_mean) = +0.082
[perm] VERDICT PERMUTATION : signal NON significatif (p >= 0.05) -- la discrimination observee est compatible avec le null intra-fold.


## Lecture 4 -- p = 0.1683 : la discrimination reste compatible avec le null

Le controle de permutation permute les 72 unites au niveau global (preservation 12H/60N), refit le pipeline intra-fold pour CHAQUE permutation, et ne stocke qu'UNE moyenne CV par permutation. Resultat : distribution nulle `0.226 +/- 0.092`, IC95 [0.046, 0.410], 16 moyennes nulles sur 100 superieures ou egales au reel — p_value fini (k+1)/(N+1) = `0.1683`.

0.308 (F1 reel) est au-dedans du nuage null : la difference reel - null = +0.082, **inferieure a un ecart-type null** (0.092). Le verdict de permutation est donc sans appel : la discrimination observee est compatible avec le hasard intra-fold, et le p fini reste loin du seuil 0.05. C'est la mesure la plus solide du notebook (100 repetitions, 3 folds chacune).

Ce que la taille a change, et ce qu'elle n'a pas change. Le p passe de `0.3267` (36 unites, 6 H) a `0.1683` (72 unites, 12 H) : un progres dans la direction attendue — plus d'unites, moins de variance du null, donc un p plus petit — mais un p qui reste **trois fois au-dessus** du seuil. C'est exactement ce que le cadran de #26829 predit : a 72 unites, cette sonde voit un effet de `d >= 2.0`, pas un effet de taille moyenne. La taille a donc **elargi la borne**, elle n'a pas change le verdict.


In [10]:
# Controle 2 -- SURFACE-ONLY sur les groupes H uniques, RAPPEL/SENSIBILITE
# (y_true=1) + delta OOF de probabilite positive original -> strippe sur les H
# uniques avec IC.
# Deux choix de mesure :
# - y_true est tout-1 sur les H test -> utiliser rappel/sensibilite, pas F1.
# - rapporter delta_proba OOF sur les H uniques (avec IC) = mesure informative
#   de combien le strip des marqueurs humoristiques fait chuter la probabilite
#   que le classifieur attribue la classe 1.
from sklearn.metrics import recall_score

def strip_humor_markers(text):
    """Surface-only : retire ponctuation expressive, verbes de recadrage, callbacks."""
    text = re.sub(r"[—«»“”…]", "", text)
    tokens = re.findall(r"[A-Za-zÀ-ÿÄ-Å']+|[,.]", text)
    out = []
    for t in tokens:
        if t.lower() in CALLBACK_LEX:
            continue
        out.append(t)
    return " ".join(out)

# Pour chaque H unique :
# - X_h_original = moyenne des repetitions (deja dans X_group si h_key en tete).
# - X_h_surface = moyenne des repetitions surface-only.
# - p_original = y_proba_all[h_idx, 1] (OOF baseline).
# - p_surface = pipe.predict_proba(X_h_surface.reshape(1,-1))[0, 1] (modele refit intra-fold).
h_keys_unique = list(H_KEY_TO_POSITIONS.keys())
surface_p_pos = []
original_p_pos = []
rappel_per_fold = []
n_h_per_fold = []

for fold_idx, (train_idx, test_idx) in enumerate(SPLITS):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, class_weight='balanced')),
    ])
    pipe.fit(X_group[train_idx], y_group[train_idx])
    test_h_mask = (y_group[test_idx] == 1)
    test_h_indices = test_idx[test_h_mask]
    n_h_per_fold.append(len(test_h_indices))
    y_true_h = np.ones(len(test_h_indices), dtype=int)
    y_pred_h = pipe.predict(X_group[test_h_indices])
    recall = recall_score(y_true_h, y_pred_h, zero_division=0)
    rappel_per_fold.append(recall)
    print(f"[surf] fold {fold_idx}  n_H_test={len(test_h_indices)}  rappel_baseline={recall:.3f}")
    # Surface : strip les repetitions des H uniques en test, moyenne par H.
    for h_idx in test_h_indices:
        # h_idx est l'index dans X_group ; retrouver l'instance source.
        h_key = h_keys_unique[h_idx]   # les groupes H occupent les premieres positions
        positions_in_paires = H_KEY_TO_POSITIONS[h_key]
        instances_h = [instances_paires[p] for p in positions_in_paires]
        # Strip + vec + moyenne.
        surf_texts = [strip_humor_markers(inst['texte']) for inst in instances_h]
        surf_instances = []
        for inst, st in zip(instances_h, surf_texts):
            inst_s = dict(inst)
            inst_s['texte'] = st
            surf_instances.append(inst_s)
        X_h_surf = np.array([vec(i) for i in surf_instances]).mean(axis=0).reshape(1, -1)
        p_surf = pipe.predict_proba(X_h_surf)[0, 1]
        p_orig = y_proba_all[h_idx, 1]
        original_p_pos.append(p_orig)
        surface_p_pos.append(p_surf)

original_p_pos = np.array(original_p_pos)
surface_p_pos = np.array(surface_p_pos)
delta_proba_oof = original_p_pos - surface_p_pos   # baisse attendue si strip = info humoristique
recall_baseline = np.mean(rappel_per_fold)
print(f"[surf] rappel (sensibilite) baseline sur H test = {recall_baseline:.3f} +/- {np.std(rappel_per_fold):.3f}")
print(f"[surf] P(y=1) original OOF sur H = {original_p_pos}")
print(f"[surf] P(y=1) surface OOF sur H = {np.round(surface_p_pos, 3)}")
print(f"[surf] delta_proba OOF (original - surface) par H : {np.round(delta_proba_oof, 3)}")
print(f"[surf] delta_proba OOF mean = {delta_proba_oof.mean():.3f} +/- {delta_proba_oof.std():.3f}")
delta_ic95_low = np.percentile(delta_proba_oof, 2.5)
delta_ic95_high = np.percentile(delta_proba_oof, 97.5)
print(f"[surf] delta_proba OOF IC95 = [{delta_ic95_low:.3f}, {delta_ic95_high:.3f}]")
if delta_proba_oof.mean() > 0 and delta_ic95_low > 0:
    print("[surf] VERDICT SURFACE : strip abaisse significativement P(y=1) -> le contenu strippe porte l'information humoristique.")
elif delta_proba_oof.mean() < 0:
    print("[surf] VERDICT SURFACE : strip AUGMENTE P(y=1) -> la surface (marqueurs retires) est contre-informative.")
else:
    print("[surf] VERDICT SURFACE : strip n'a pas d'effet clair (IC95 traverse 0) -- discrimination purement lexicale de surface.")


[surf] fold 0  n_H_test=4  rappel_baseline=0.250
[surf] fold 1  n_H_test=4  rappel_baseline=0.500
[surf] fold 2  n_H_test=4  rappel_baseline=0.500
[surf] rappel (sensibilite) baseline sur H test = 0.417 +/- 0.118
[surf] P(y=1) original OOF sur H = [0.85649922 0.33573622 0.41037102 0.35379631 0.78457604 0.12020868
 0.66030173 0.15703061 0.42111732 0.51953007 0.39349203 0.70759661]
[surf] P(y=1) surface OOF sur H = [0.628 0.51  0.564 0.504 0.34  0.112 0.539 0.101 0.421 0.523 0.416 0.525]
[surf] delta_proba OOF (original - surface) par H : [ 0.228 -0.174 -0.154 -0.151  0.445  0.009  0.122  0.056  0.    -0.004
 -0.022  0.182]
[surf] delta_proba OOF mean = 0.045 +/- 0.172
[surf] delta_proba OOF IC95 = [-0.169, 0.385]
[surf] VERDICT SURFACE : strip n'a pas d'effet clair (IC95 traverse 0) -- discrimination purement lexicale de surface.


## Lecture 5 -- Le strip : IC95 [-0.169, +0.385], aucun verdict

Le second controle retire les marqueurs humoristiques de surface (ponctuation expressive, verbes de recadrage, callbacks) puis re-mesure la probabilite positive OOF. Resultats : rappel baseline sur H test = `0.417 +/- 0.118` (folds 0.250 / 0.500 / 0.500) ; delta de probabilite original -> strippe = `+0.045 +/- 0.172` par H, IC95 `[-0.169, 0.385]`.

L'IC95 traverse 0, et la cellule le dit explicitement : sur 12 H, un intervalle qui traverse 0 est attendu — il ne PROUVE ni que la surface porte l'information humoristique, ni qu'elle ne la porte pas. La cellule declare donc « strip n'a pas d'effet clair » et refuse le verdict SURFACE_SEULE qui aurait ete abusif (la cellule de verdict le rappelle dans ses limites). Coherent avec l'ensemble du carnet : trois mesures, trois non-conclusions honnetes.

A comparer au rejeu a 6 groupes H (delta `+0.123 +/- 0.305`, IC95 `[-0.222, 0.588]`) : le delta moyen a **decrease** en passant a 12 groupes H, et l'IC95 s'est **resserre** (de `[-0.222, 0.588]` a `[-0.169, 0.385]`) tout en continuant a contenir 0. Plus de donnees a donc reduit l'incertitude sans faire emerger de direction — l'intervalle se resserre autour d'un point qui reste compatible avec zero.


In [11]:
# L'ACCORD DE LA CHAINE DE QUALIFICATION, lu dans l'ARTEFACT COMMITE (#26924).
# Trois quantites a ne pas confondre -- c'est precisement la confusion que
# cette cellule leve :
#   * kappa          : accord CORRIGE DU HASARD. Il retire l'accord que le
#                      hasard produirait sur ces marges ; 1 - kappa n'est donc
#                      PAS un taux de desaccord.
#   * accord observe : taux d'accord BRUT. Le desaccord brut vaut
#                      1 - accord observe, compte en items.
#   * desaccord brut : le nombre reel d'items mal classes.
# Ce que le kappa NE donne pas : un plafond de F1. Un tel plafond supposerait
# de dire quelle etiquetage fait foi, et la mesure ne le dit pas. On publie
# donc les trois quantites, la matrice, et la lecture par classe.
RAPPORT_ACCORD = Path("MyIA.AI.Notebooks/IIT/ICT-Series/ict/results/humor_llm_chain_c26911.json")
assert RAPPORT_ACCORD.exists(), f"rapport d'accord manquant : {RAPPORT_ACCORD}"
_rapport = json.loads(RAPPORT_ACCORD.read_text(encoding="utf-8"))
_origine = _rapport["measures"]["vs_reference_by_source"]["par_origine_du_label"]
ACC = _origine["manuel"]                       # items a etiquette humaine (n = 60)
ARG = _origine["heuristique-argumentum"]       # items Argumentum (n = 15)


def _matrice(population):
    """Matrice de confusion : lignes = chaine (modele), colonnes = reference.

    L'orientation n'est pas supposee : elle est verifiee sur les marges que
    l'instrument publie ('a' = la chaine, 'b' = la reference humaine).
    """
    cats = population["categories"]
    table = {c: {c2: population["confusion"].get(c, {}).get(c2, 0) for c2 in cats}
             for c in cats}
    n = sum(sum(r.values()) for r in table.values())
    diag = sum(table[c][c] for c in cats)
    return cats, table, n, diag


ACC_CATS, ACC_TAB, ACC_N, ACC_DIAG = _matrice(ACC)
_marge_chaine = {c: sum(ACC_TAB[c].values()) / ACC_N for c in ACC_CATS}
_marge_ref = {c: sum(ACC_TAB[c2][c] for c2 in ACC_CATS) / ACC_N for c in ACC_CATS}
assert all(abs(_marge_chaine[c] - ACC["marginals"]["a"][c]) < 1e-9 for c in ACC_CATS), \
    "orientation de la matrice inattendue (marge 'a' <> chaine)"
assert all(abs(_marge_ref[c] - ACC["marginals"]["b"][c]) < 1e-9 for c in ACC_CATS), \
    "orientation de la matrice inattendue (marge 'b' <> reference)"

ACC_KAPPA = float(ACC["kappa"])
ACC_OBS = float(ACC["observed_agreement"])
ACC_EXP = float(ACC["expected_agreement"])
ACC_DESACCORD_N = ACC_N - ACC_DIAG
ACC_DESACCORD_PCT = 100 * ACC_DESACCORD_N / ACC_N

# Classe positive : la chaine contre la reference humaine.
ACC_POS = "humour_reussi"
ACC_TP = ACC_TAB[ACC_POS][ACC_POS]
ACC_CHAIN_POS = sum(ACC_TAB[ACC_POS].values())
ACC_REF_POS = sum(ACC_TAB[c][ACC_POS] for c in ACC_CATS)
ACC_FP = ACC_CHAIN_POS - ACC_TP
ACC_FN = ACC_REF_POS - ACC_TP
ACC_SENS = ACC_TP / ACC_REF_POS
ACC_PREC = ACC_TP / ACC_CHAIN_POS
ACC_F1 = 2 * ACC_TP / (ACC_REF_POS + ACC_CHAIN_POS)

# Population Argumentum : l'etiquette n'y est pas derivable du texte tronque.
ARG_CATS, ARG_TAB, ARG_N, ARG_DIAG = _matrice(ARG)
ARG_OBS = float(ARG["observed_agreement"])
ARG_EXP = float(ARG["expected_agreement"])
ARG_KAPPA = float(ARG["kappa"])

print(f"[accord] artefact commite : {RAPPORT_ACCORD.name}")
print(f"[accord] population a etiquette humaine : n = {ACC_N}")
print(f"[accord]   kappa (accord corrige du hasard) : {ACC_KAPPA:.3f}")
print(f"[accord]   accord observe (brut)            : {ACC_OBS:.3f}")
print(f"[accord]   accord attendu (par hasard)      : {ACC_EXP:.3f}")
print(f"[accord]   desaccord BRUT                   : {ACC_DESACCORD_N} items sur {ACC_N} = {ACC_DESACCORD_PCT:.1f} %")
print(f"[accord]   1 - kappa = {100 * (1 - ACC_KAPPA):.1f} % n'est PAS un taux de desaccord : c'est l'accord que la correction du hasard retire")
print("[accord] matrice (lignes = chaine, colonnes = reference humaine) :")
for _c in ACC_CATS:
    print("[accord]   " + f"{_c:28s}" + " ".join(f"{_c2[:9]:>9s}={ACC_TAB[_c][_c2]:2d}" for _c2 in ACC_CATS))
print(f"[accord] classe positive '{ACC_POS}' : reference {ACC_REF_POS} positifs, chaine {ACC_CHAIN_POS} positifs, "
      f"concordants {ACC_TP}, non positifs pour la chaine {ACC_FP}, positifs humains manques {ACC_FN}")
print(f"[accord]   sensibilite {ACC_SENS:.3f} | precision {ACC_PREC:.3f} | F1 de la chaine (vs reference) {ACC_F1:.3f}")
print(f"[accord] population Argumentum (n = {ARG_N}) : accord observe {ARG_OBS:.3f} = accord attendu {ARG_EXP:.3f}, kappa = {ARG_KAPPA:.3f}")
print("[accord]   (l'etiquette Argumentum n'est pas derivable du texte tronque -- cf #26924)")

# Origine reellement consommee par ce carnet : les groupes H et les negatifs ne
# viennent pas tous de la meme origine d'etiquette (humaine ou heuristique).
H_ORIGINE = Counter(_h.get("source", "?") for _h in H_REUSSI)
N_ORIGINE = Counter(_n.get("source", "?") for _h, _n, _d in PAIRES_EXPL)
print(f"[accord] origine des {len(H_REUSSI)} groupes H de ce carnet : {dict(H_ORIGINE)}")
print(f"[accord] origine des {len(PAIRES_EXPL)} negatifs de ce carnet : {dict(N_ORIGINE)}")

[accord] artefact commite : humor_llm_chain_c26911.json
[accord] population a etiquette humaine : n = 60
[accord]   kappa (accord corrige du hasard) : 0.863
[accord]   accord observe (brut)            : 0.917
[accord]   accord attendu (par hasard)      : 0.390
[accord]   desaccord BRUT                   : 5 items sur 60 = 8.3 %
[accord]   1 - kappa = 13.7 % n'est PAS un taux de desaccord : c'est l'accord que la correction du hasard retire
[accord] matrice (lignes = chaine, colonnes = reference humaine) :
[accord]   humour_reussi               humour_re=29 rire_sans= 0 recadrage= 3 offensif_= 0      rien= 0
[accord]   rire_sans_recadrage         humour_re= 0 rire_sans= 3 recadrage= 0 offensif_= 1      rien= 0
[accord]   recadrage_sans_rire         humour_re= 0 rire_sans= 0 recadrage= 0 offensif_= 0      rien= 0
[accord]   offensif_compris_non_partagehumour_re= 0 rire_sans= 0 recadrage= 0 offensif_= 3      rien= 0
[accord]   rien                        humour_re= 1 rire_sans= 0 recadrage

## Lecture 5 bis -- L'accord mesuré, et ce qu'il ne dit pas

La limite n° 5 du verdict (cellule suivante) encadre l'accord de la chaîne de qualification avec une référence humaine. La cellule précédente lit l'artefact commité de cette mesure (`ict/results/humor_llm_chain_c26911.json`, campagne v2 de #26924) et en publie les trois quantités — **elles ne sont pas équivalentes** :

| Quantité | Valeur mesurée | Ce que c'est |
|---|---|---|
| `kappa` | 0,863 | accord **corrigé du hasard** : il retire l'accord que le hasard produirait sur ces marges (accord attendu 0,390) |
| accord observé | 0,917 | taux d'accord **brut** sur les 60 items à étiquette humaine (55 items sur 60) |
| désaccord brut | **5 items sur 60 (8,3 %)** | le compte réel des items mal classés, sur les **cinq classes** |

Le « 1 − kappa ≈ 13,7 % » n'est **ni** un taux d'accord brut **ni** un taux de désaccord : c'est la part d'accord que la correction du hasard retire. Le taux de désaccord brut est de 8,3 %.

Sur la **classe positive** (`humour_reussi`), la matrice donne des nombres distincts : la chaîne étiquette 32 items positifs, la référence humaine n'en confirme que 29 — **3 (9,4 %) ne sont pas des positifs humains** — et la chaîne laisse de côté **1 des 30** positifs humains. Sur cette classe, la chaîne vaut une sensibilité de 0,967, une précision de 0,906 et un **F1 de 0,935** face à la référence humaine. Le « 14 % de désaccord sur la classe positive » que cette lecture portait auparavant ne correspond à aucune de ces quantités.

**Ce que cette mesure n'établit pas.** Le κ ne fournit **aucun plafond de F1** pour la sonde : un tel plafond supposerait de dire quel étiquetage fait foi, et la mesure ne le dit pas. Ce qui est mesuré, c'est l'accord entre **une** labellisation de la taxonomie — celle de la chaîne LLM de #26911 étape 2 — et la référence humaine. Sur la population `Argumentum`, la même mesure vaut un accord observé de 0,200 **exactement égal** à l'accord attendu : l'étiquette n'y est pas dérivable du texte tronqué (#26924).

**L'origine réellement consommée par ce carnet.** La même cellule compte, sur les unités que le carnet mesure : les **12 groupes H** retenus se répartissent en **5 `blague-manuelle` (annotation humaine) et 7 `Argumentum` (étiquette heuristique)** ; les **60 négatifs** en **21 humains** (13 `declaration-factuelle` + 8 `edge-case-curated`) et **39 `Argumentum`**. La classe positive de ce carnet est donc majoritairement **heuristique** : la discrimination mesurée porte, pour 7 de ses 12 positifs et 39 de ses 60 négatifs, sur la distinction **interne à l'heuristique Argumentum** — c'est-à-dire sur des paires dont l'étiquette positive n'est pas dérivable du texte (cf. #26924). Cela ne change aucun verdict (le verdict était déjà `INCONCLUSIVE borne`), mais cela en déplace la portée : le `F1 = 0.308` est une mesure de discrimination **interne à l'étiquetage** bien plus qu'une mesure de l'humour au sens humain, et c'est la lecture que la limite n° 3 du verdict (« Labellisation Argumentum = heuristique, pas or humain ») doit porter.

In [12]:
# VERDICT FERME -- issu des 3 mesures group-level :
# 1. F1 group-level baseline
# 2. Distribution nulle via 100 permutations INTRA-FOLD de y_group :
#    1 moyenne CV par permutation, p fini (k+1)/(N+1).
# 3. Rappel baseline + delta_proba OOF original -> strippe sur les groupes H

print("=" * 70)
print("VERDICT -- Etage 0 lexical baseline ICT-35 (GROUP-LEVEL)")
print("=" * 70)
print(f"  F1 group-level baseline           = {f1_real:.3f} +/- {f1_real_std:.3f}")
print(f"  F1 null distribution (100 perms)  = mean {perm_mean:.3f} +/- {perm_std:.3f}")
print(f"  F1 null IC95                      = [{perm_ic95_low:.3f}, {perm_ic95_high:.3f}]")
print(f"  p_value fini (k+1)/(N+1)          = {p_value_perm:.4f}")
print(f"  null >= reel                      = {n_above_or_equal}/100")
print(f"  Rappel (sensibilite) baseline H   = {recall_baseline:.3f}")
print(f"  delta_proba OOF H (orig - surf)   = {delta_proba_oof.mean():.3f} +/- {delta_proba_oof.std():.3f}")
print(f"  delta_proba OOF H IC95            = [{delta_ic95_low:.3f}, {delta_ic95_high:.3f}]")
print()
# Regle de decision ajustee pour p fini (k+1)/(N+1).
# - p_value fini < 0.05 ET delta_proba > 0 avec IC95 > 0 : FEATURE_CANDIDATE.
# - p_value fini >= 0.05 : INCONCLUSIVE borne (la discrimination observee est
#   compatible avec le null intra-fold ; le verdict ne peut etre plus precis que
#   "borne sur ce sous-corpus").
# - delta_proba <= 0 OU IC95 traverse 0 : PAS un verdict SURFACE_SEULE (l'IC
#   sur les groupes H traverse 0 n'est PAS la preuve d'une discrimination
#   purement lexicale de surface -- sur un petit nombre de groupes H, IC95
#   traverse 0 trivialement ; SURFACE_SEULE demanderait un IC95 entierement
#   >= 0 avec un N suffisant). On documente la mesure et on la declare
#   "delta_proba IC95 traverse 0" sans trancher.
if p_value_perm < 0.05 and delta_proba_oof.mean() > 0 and delta_ic95_low > 0:
    verdict = "FEATURE_CANDIDATE"
    why = (
        "p_value fini < 0.05 ET delta_proba OOF > 0 avec IC95 > 0 : la discrimination "
        "est significative au-dela du null intra-fold ET le strip de la surface humoristique "
        "fait chuter P(y=1) de maniere fiable. Le signal lexical depasse la surface strippee."
    )
elif p_value_perm >= 0.05:
    verdict = "INCONCLUSIVE borne"
    why = (
        f"p_value fini = {p_value_perm:.4f} (k+1)/(N+1) >= 0.05 : la discrimination observee "
        f"est compatible avec le null intra-fold (100 permutations, 1 moyenne CV par perm, "
        f"mean {perm_mean:.3f}). Borne sur ce sous-corpus ({n_total} unites group-level, "
        f"{n_h} H uniques) -- pas une falsification de l'humour lexical en general. Le verdict "
        f"ne peut etre plus precis que 'borne sur ce sous-corpus' avec ce N et 6 features."
    )
else:
    verdict = "INCONCLUSIVE borne"
    why = (
        f"p_value fini = {p_value_perm:.4f} < 0.05 mais delta_proba OOF IC95 = "
        f"[{delta_ic95_low:.3f}, {delta_ic95_high:.3f}] traverse 0 : avec {n_h} groupes H, "
        f"un IC95 qui traverse 0 est attendu et n'invalide PAS la discrimination. "
        f"On reste sur INCONCLUSIVE borne."
    )
print(f"  VERDICT = {verdict}")
print(f"  RAISON  = {why}")
print()
print("=" * 70)
print("LIMITES DE CE VERDICT (reformulees -- ce n'est PAS une falsification)")
print("=" * 70)
print(f"1. {n_total} unites group-level ({n_h} H uniques + {n_total - n_h} N distincts) -- la taille que le cadran de #26829 exige pour voir un effet de d >= 2.0 a 80 % de puissance.")
print("2. Verdict borne a ces 6 features (n_tokens, ponctuation, verbes, callbacks, stop, repetition).")
print("3. Labellisation Argumentum = heuristique, pas or humain. Cf note methodo #14035.")
print("4. Permutation intra-fold : 1 moyenne CV / perm, p fini (k+1)/(N+1).")
print(f"5. 1 seul modele LLM (qwen3.6-35b-a3b) sert a labelliser ; le verdict est local a ce couple. "
      f"L'accord de ce modele avec une reference humaine, mesure par la chaine de qualification (#26911 etape 2), "
      f"est un accord CORRIGE DU HASARD : kappa = {ACC_KAPPA:.3f} (accord observe {ACC_OBS:.3f}, "
      f"accord attendu {ACC_EXP:.3f}), soit {ACC_DESACCORD_N} items en desaccord sur {ACC_N} "
      f"({ACC_DESACCORD_PCT:.1f} %). Le kappa ne fixe AUCUN plafond de F1 : le bruit d'annotation "
      f"borne le signal que cette sonde peut chercher.")
print(f"6. Pipeline intra-fold integre + StratifiedKFold(n_splits=3) sur {n_total} unites group-level.")
print(f"7. delta_proba sur {n_h} groupes H : IC95 traverse 0 sur un nombre de groupes H qui reste faible ; on declare 'IC traverse 0'")
print("   sans pretendre que c'est la preuve d'une discrimination purement lexicale de surface.")


VERDICT -- Etage 0 lexical baseline ICT-35 (GROUP-LEVEL)
  F1 group-level baseline           = 0.308 +/- 0.156
  F1 null distribution (100 perms)  = mean 0.226 +/- 0.092
  F1 null IC95                      = [0.046, 0.410]
  p_value fini (k+1)/(N+1)          = 0.1683
  null >= reel                      = 16/100
  Rappel (sensibilite) baseline H   = 0.417
  delta_proba OOF H (orig - surf)   = 0.045 +/- 0.172
  delta_proba OOF H IC95            = [-0.169, 0.385]

  VERDICT = INCONCLUSIVE borne
  RAISON  = p_value fini = 0.1683 (k+1)/(N+1) >= 0.05 : la discrimination observee est compatible avec le null intra-fold (100 permutations, 1 moyenne CV par perm, mean 0.226). Borne sur ce sous-corpus (72 unites group-level, 12 H uniques) -- pas une falsification de l'humour lexical en general. Le verdict ne peut etre plus precis que 'borne sur ce sous-corpus' avec ce N et 6 features.

LIMITES DE CE VERDICT (reformulees -- ce n'est PAS une falsification)
1. 72 unites group-level (12 H uniques + 60

## Lecture 6 -- Le verdict ferme : INCONCLUSIVE borne, et ce qu'il borne

La cellule de verdict n'a pas de decision a chercher : `p_value fini 0.1683` >= 0.05 — le premier terme de la regle de decision tombe, le verdict est `INCONCLUSIVE borne` par construction (RAISON imprimee). Les 7 limites affichees ne sont pas une parade : elles delimitent le perimetre exact du resultat — 72 unites group-level, 12 groupes H, 6 features lexicales, labellisation par la chaine de qualification Argumentum dont l'accord avec une reference humaine echantillonnee est mesure dans l'artefact commite (#26924) : **kappa = 0.863** (accord corrige du hasard ; accord observe 0.917, desaccord brut 5 items sur 60), permutation intra-fold correcte, pipeline intra-fold.

La limite de circularite, qu'il faut nommer explicitement : la classe positive de ce carnet est **definie par la meme chaine de labellisation** dont on mesure l'accord. Le bruit d'annotation -- mesure sur 60 items a etiquette humaine : **desaccord brut de 5 items sur 60 (8,3 %)**, sur les cinq classes et non sur la seule classe positive -- borne le signal que la sonde peut chercher : un F1 mesure ici se compare au **null intra-fold**, jamais a un oracle humain. Cette borne-la n'a pas ete levee par l'élargissement a 12 groupes H : le corpus etant labelise par la meme chaine, elle subsiste a toute taille. Le **kappa = 0.863** qui accompagne cette limite est un accord **corrige du hasard** (accord observe 0.917, accord attendu 0.390) : il ne dit ni le taux de desaccord, ni un plafond de F1 -- la Lecture 5 bis publie la matrice, l'origine reellement consommee, et la distinction.

La lecture la plus importante reste epistemique : ce verdict **ne falsifie pas** l'humour lexical. Il borne la discrimination de CES 6 features sur CE sous-corpus a CE stade de la taxonomie (couche 1 : jeu sonore / repetition + incongruite perceptible). Les etages superieurs (embeddings, contexte narratif, SAE — gates #5635 / #14033) sont les seuls a pouvoir trancher au-dela. Un resultat negatif proprement circonscrit est plus utilisable qu'un verdict fort indefendable : c'est la discipline que le notebook s'est imposee depuis le frontmatter.

## Le cadran de puissance de la sonde (critère D2)

La cellule de verdict conclut `INCONCLUSIVE borne`, et sa limite n° 1 énonce la raison :
« 72 unités group-level (12 H uniques + 60 N distincts) ». Mais un verdict borné dont on ne
mesure pas l'étendue de la borne reste une phrase : à 72 unités, quelle taille d'effet
cette sonde peut-elle voir ? combien d'unités auraient-il fallu pour qu'elle voie un
signal ?

C'est le critère **D2** de la qualification des substrats (#24282) : *combien de tirages
faut-il pour identifier la meilleure action ?* Appliqué ici, il ne demande pas au carnet
de changer de verdict — il demande de dire **ce que ce verdict permet et ne permet pas
d'exclure**.

La calibration qui suit ne touche pas au corpus : elle mesure l'**instrument**. On rejoue
exactement son protocole (6 features standardisées, ratio 1 H pour 5 N, `StratifiedKFold`
à 3 plis, `LogisticRegression(C=1.0, class_weight='balanced')`, F1 moyenne sur les plis)
sur une famille synthétique dont on **contrôle** la force du signal `d` : un décalage de
moyenne réparti sur les 6 features, soit `d` unités d'écart-type de séparation entre les
centres de classe. Le seuil de détection est le 95ᵉ centile de la distribution nulle
mesurée **à la même taille** (même nombre de groupes H, même ratio, mêmes plis) : la
comparaison est appariée. La puissance est la fraction des réplications où le F1 observé
dépasse ce seuil.

La limite est écrite dans la cellule, avant la mesure : cette calibration dit la
sensibilité de l'**instrument**, pas la réalité du corpus (features gaussiennes contre
comptes lexicaux croches, labels heuristiques). Elle ne bouge pas le verdict — elle en
donne l'étendue.

Le cadran balaye quatre tailles (6, 12, 24 et 48 groupes H) ; **celle du carnet est la
seconde** (12 groupes H, 72 unités), parce que c'est la taille que le corpus permet
réellement d'atteindre : au-delà, il manque les négatifs.


In [13]:
# --- Cadran de puissance de la sonde (critere D2 de la qualification des substrats, #24282) ---
# Le carnet conclut "INCONCLUSIVE borne" : la discrimination observee est compatible
# avec le null. Il ne dit pas ce que cette borne VALUT : a 72 unites group-level
# (12 groupes H), quelle taille d'effet cette sonde peut-elle detecter, et combien
# d'unites faudrait-il pour en detecter une de reference ? C'est le cout d'identification.
#
# Protocole : on calibre l'INSTRUMENT sur une famille synthetique dont on controle
# la force du signal d, en reprenant exactement le protocole du carnet (6 features
# standardisees, ratio de classe 1/6 -- 1 H pour 5 N, StratifiedKFold a 3 plis,
# LogisticRegression(C=1.0, class_weight='balanced'), F1 moyenne sur les plis).
# Le signal plante est un decalage de moyenne reparti sur les 6 features : la
# distance de Mahalanobis entre les centres de classe vaut alors d, soit d unites
# d'ecart-type de separation (une "taille d de Cohen").
#
# Le seuil de detection est le 95e centile de la distribution nulle mesuree A LA
# MEME TAILLE (meme n_h, meme ratio, memes plis) : la comparaison est appariee.
# La puissance = fraction des replications ou le F1 observe depasse ce seuil.
#
# COUT : chaque replication est une validation croisee de 3 plis sur des
# donnees synthetiques ; R_REPLICATIONS replications par case, 4 tailles, 6
# valeurs de d. Le cout est mesure et affiche (temps de calibration), pour que le
# lecteur sache ce que re-executer ce carnet lui coute.
#
# PRECISION : a R_REPLICATIONS replications, l'ecart-type de la puissance estimee
# vaut environ sqrt(p(1-p)/R) -- soit +-8 points autour de 80 %. Les valeurs de
# "plus petit d detectable" ci-dessous sont donc des BORNES a cette precision,
# pas des valeurs exactes.
#
# LIMITE HONNETE : cette calibration mesure la sensibilite de l'INSTRUMENT, pas la
# realite du corpus. La famille synthetique est gaussienne, les features reelles
# sont des comptes lexicaux (croches, correles), les labels sont heuristiques.
# Les nombres ci-dessous sont un ordre de grandeur du cout d'identification de
# CETTE sonde -- ils ne sont pas une borne sur l'humour lexical.
SEED_PUISSANCE = SEED + 4242
RATIO_N = 5                     # 5 non-H pour 1 H -- le ratio du carnet (60 N pour 12 H)
N_FEAT = 6
N_H_VALEURS = (6, 12, 24, 48)  # 12 = la taille du carnet apres rejeu ; 6 = l'ancienne taille ; les autres = combien il faudrait
D_VALEURS = (0.5, 1.0, 1.5, 2.0, 2.5, 3.0)
R_REPLICATIONS = 100
PUISSANCE_CIBLE = 0.80
PRECISION_PUISSANCE = 1.96 * (0.8 * 0.2 / R_REPLICATIONS) ** 0.5


def simule_groupes(n_h, d, rng):
    """n_h groupes H, RATIO_N non-H par H ; decalage de moyenne de d sur les 6 features."""
    n = n_h * (1 + RATIO_N)
    X = rng.normal(size=(n, N_FEAT))
    y = np.zeros(n, dtype=int)
    positions_h = rng.choice(n, n_h, replace=False)
    y[positions_h] = 1
    # Decalage reparti sur les 6 features : distance de Mahalanobis = d.
    X[positions_h] += d / np.sqrt(N_FEAT)
    return X, y


def f1_moyenne_cv(X, y):
    """Le protocole du carnet : 3 plis stratifies, pipeline intra-pli, F1 moyenne."""
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    f1s = []
    for train_idx, test_idx in skf.split(X, y):
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED,
                                      class_weight='balanced')),
        ])
        pipe.fit(X[train_idx], y[train_idx])
        y_pred = pipe.predict(X[test_idx])
        f1s.append(f1_score(y[test_idx], y_pred, zero_division=0))
    return float(np.mean(f1s))


print(f"[puissance] calibration de l'instrument | {R_REPLICATIONS} replications par case")
print(f"[puissance] ratio 1 H pour {RATIO_N} N | {N_FEAT} features | seuil = 95e centile du null a la meme taille")
print(f"[puissance] puissance cible = {PUISSANCE_CIBLE:.0%} | precision sur la puissance "
      f"= +/- {PRECISION_PUISSANCE:.1%}")
print()

t_calibration = time.perf_counter()

puissance_table = {}    # (n_h, d) -> puissance
f1_null_par_nh = {}     # n_h -> tableau des F1 nuls
d_min_par_nh = {}       # n_h -> plus petit d ou la puissance atteint la cible

for n_h in N_H_VALEURS:
    n_total = n_h * (1 + RATIO_N)
    # --- null : aucun signal, meme taille, meme ratio ---
    rng_null = np.random.default_rng(SEED_PUISSANCE + n_h)
    f1_null = np.array([f1_moyenne_cv(*simule_groupes(n_h, 0.0, rng_null))
                        for _ in range(R_REPLICATIONS)])
    f1_null_par_nh[n_h] = f1_null
    seuil = float(np.percentile(f1_null, 95))
    print(f"=== n_h = {n_h} groupes H ({n_total} unites) ===")
    print(f"  null (d=0) : F1 moyen = {f1_null.mean():.3f} +/- {f1_null.std():.3f} | "
          f"95e centile = {seuil:.3f} | 5e centile = {np.percentile(f1_null, 5):.3f}")
    d_min_par_nh[n_h] = None
    for d in D_VALEURS:
        rng = np.random.default_rng(SEED_PUISSANCE + 1000 * n_h + int(10 * d))
        f1s = np.array([f1_moyenne_cv(*simule_groupes(n_h, d, rng))
                        for _ in range(R_REPLICATIONS)])
        puissance = float((f1s > seuil).mean())
        puissance_table[(n_h, d)] = puissance
        if d_min_par_nh[n_h] is None and puissance >= PUISSANCE_CIBLE:
            d_min_par_nh[n_h] = d
        print(f"  d = {d:.1f} : F1 moyen = {f1s.mean():.3f} +/- {f1s.std():.3f} | "
              f"puissance = {puissance:.2f}")
    print()

print("=" * 78)
print("COUT D'IDENTIFICATION -- plus petit d detecte a puissance >= "
      f"{PUISSANCE_CIBLE:.0%} (et les unites qu'il faut)")
print("=" * 78)
print("n_h (groupes H) | n (unites group-level) | d_min detectable")
for n_h in N_H_VALEURS:
    d_min = d_min_par_nh[n_h]
    d_affiche = f"{d_min:.1f}" if d_min is not None else "> 3.0 (hors grille)"
    print(f"{n_h:16d} | {n_h * (1 + RATIO_N):22d} | {d_affiche}")

print()
for d_ref in (1.0, 2.0):
    n_h_requis = next((n_h for n_h in N_H_VALEURS
                       if puissance_table[(n_h, d_ref)] >= PUISSANCE_CIBLE), None)
    if n_h_requis is None:
        print(f"effet de reference d = {d_ref:.1f} : non atteint a puissance "
              f"{PUISSANCE_CIBLE:.0%} dans la grille (au-dela de {max(N_H_VALEURS)} groupes H)")
    else:
        print(f"effet de reference d = {d_ref:.1f} : il faut {n_h_requis} groupes H "
              f"({n_h_requis * (1 + RATIO_N)} unites group-level) pour atteindre "
              f"{PUISSANCE_CIBLE:.0%} de puissance ; le carnet en a {N_H_GROUPES}")

# --- ou le resultat reel se situe-t-il dans le null sans signal ? ---
null_pilote = f1_null_par_nh[N_H_GROUPES]
# rang_reel = fraction des nuls inferieurs OU EGAUX a la valeur observee : la valeur
# observee est donc au rang rang_reel de la distribution nulle. Le compte affiche
# est le meme (inferieurs ou egaux) et le complement exact (strictement au-dessus) :
# les deux nombres somment a len(null_pilote), et la phrase affichee ne peut donc
# pas s'inverser.
rang_reel = float((null_pilote <= f1_real).mean())
n_null_sous_ou_egal = int((null_pilote <= f1_real).sum())
n_null_au_dessus = int((null_pilote > f1_real).sum())
print()
print(f"[puissance] le F1 reel du carnet ({f1_real:.3f}) se situe au quantile "
      f"{rang_reel:.0%} de la distribution nulle SANS signal a la taille du carnet "
      f"({N_H_GROUPES} groupes H : {n_null_sous_ou_egal}/{len(null_pilote)} nuls sont en dessous "
      f"ou egaux, {n_null_au_dessus}/{len(null_pilote)} le depassent)")
print("[puissance] lecture : la valeur observee est ce que l'instrument produit quand "
      "il n'y a rien -- elle ne peut donc pas etre lue comme un signal faible.")
print(f"[puissance] cout de la calibration : {time.perf_counter() - t_calibration:.1f} s")


[puissance] calibration de l'instrument | 100 replications par case
[puissance] ratio 1 H pour 5 N | 6 features | seuil = 95e centile du null a la meme taille
[puissance] puissance cible = 80% | precision sur la puissance = +/- 7.8%



=== n_h = 6 groupes H (36 unites) ===
  null (d=0) : F1 moyen = 0.177 +/- 0.141 | 95e centile = 0.420 | 5e centile = 0.000


  d = 0.5 : F1 moyen = 0.249 +/- 0.146 | puissance = 0.14


  d = 1.0 : F1 moyen = 0.308 +/- 0.192 | puissance = 0.25


  d = 1.5 : F1 moyen = 0.455 +/- 0.168 | puissance = 0.51


  d = 2.0 : F1 moyen = 0.580 +/- 0.193 | puissance = 0.78


... (19 outputs left out)

  d = 1.5 : F1 moyen = 0.512 +/- 0.052 | puissance = 1.00


  d = 2.0 : F1 moyen = 0.632 +/- 0.051 | puissance = 1.00


  d = 2.5 : F1 moyen = 0.735 +/- 0.048 | puissance = 1.00


  d = 3.0 : F1 moyen = 0.816 +/- 0.044 | puissance = 1.00

COUT D'IDENTIFICATION -- plus petit d detecte a puissance >= 80% (et les unites qu'il faut)
n_h (groupes H) | n (unites group-level) | d_min detectable
               6 |                     36 | 2.5
              12 |                     72 | 2.0
              24 |                    144 | 1.5
              48 |                    288 | 1.0

effet de reference d = 1.0 : il faut 48 groupes H (288 unites group-level) pour atteindre 80% de puissance ; le carnet en a 12
effet de reference d = 2.0 : il faut 12 groupes H (72 unites group-level) pour atteindre 80% de puissance ; le carnet en a 12

[puissance] le F1 reel du carnet (0.308) se situe au quantile 72% de la distribution nulle SANS signal a la taille du carnet (12 groupes H : 72/100 nuls sont en dessous ou egaux, 28/100 le depassent)
[puissance] lecture : la valeur observee est ce que l'instrument produit quand il n'y a rien -- elle ne peut donc pas etre lue comme un signal

### Lecture 7 -- ce que le cadran dit, et ce qu'il ne dit pas

Trois résultats.

**1. Le coût d'identification de cette sonde, à 80 % de puissance.**

| groupes H | unités group-level | plus petit effet détectable |
|---|---|---|
| 6 (ancienne version du carnet) | 36 | d ≥ 2.5 |
| **12 (le carnet après rejeu)** | **72** | **d ≥ 2.0** |
| 24 | 144 | d ≥ 1.5 |
| 48 | 288 | d ≥ 1.0 |

Un effet de référence de **d = 1** (un écart-type de séparation entre les classes)
exigerait **48 groupes H**, soit 288 unités group-level : le carnet en a 12. Pour un effet de
**d = 2**, il en faut 12 (72 unités) : le carnet en a 12, et il l'atteint — le cadran mesure
à cette taille une puissance de **94 %** pour `d = 2.0` (et 79 % pour `d = 1.5`). Autrement
dit, à 72 unités, cette sonde voit les effets **deux fois plus forts** que l'effet de
référence ; son verdict `INCONCLUSIVE` ne peut donc toujours pas exclure un signal de
taille moyenne.

**1 bis. La taille demandée par le cadran, et ce que le corpus permet de l'atteindre.**
La ligne « 24 groupes H / 144 unités » est hors d'atteinte avec le corpus actuel : elle
exigerait **120 négatifs uniques** (5 N par groupe H), et le corpus n'en compte que **72**.
Le maximum atteignable est donc **14 groupes H (84 unités group-level)**. Le carnet mesure les
**12 groupes H** du rejeu (72 unités) : une taille **atteignable, mais en deçà de ce plafond** —
12 est la grille retenue par le protocole, 14 le plafond du corpus, et les deux chiffres ne se
confondent pas. L'écart avec la taille demandée est un fait de corpus, pas une négligence de
protocole — c'est la voie #14033 qui le referme.

**2. Deux nulls, deux mesures de « rien » — à ne pas confondre.** Le **null simulé du cadran**
(famille synthétique, calibrée à la taille du carnet : 12 groupes H) place le F1 réel (0.308)
au **72ᵉ centile** : **72 des 100 tirages nuls sont en dessous ou égaux** à la valeur observée,
**28 la dépassent**. La valeur observée n'est donc pas dans la queue de cette distribution :
elle est **au-dessus de la médiane** de ce null simulé. Le **null de permutation sur le corpus
réel** (100 permutations intra-fold de `y_group`) donne, lui, une autre mesure de « rien » :
**16 moyennes nulles sur 100 atteignent ou dépassent le réel**, soit `p = 0.1683` et un écart
`+0.082` — **inférieur à un écart-type de ce null**. Les deux nulls ne s'additionnent pas et ne
se remplacent pas : l'un est synthétique et calibré sur la taille du carnet, l'autre est
construit sur les étiquettes réelles. Ils disent la même chose, qui est ce que le carnet
énonce déjà : la discrimination observée est compatible avec le null, elle n'est pas un
signal faible.

**3. Ce que la calibration ne dit pas.** Elle mesure la **sensibilité de l'instrument**, pas
la réalité du corpus : la famille synthétique est gaussienne, les features réelles sont des
comptes lexicaux (croches, corrélés), les labels viennent de l'heuristique Argumentum, et
le critère de détection (F1 au-dessus du 95ᵉ centile du null synthétique) n'est pas
exactement le test de permutation du carnet — c'en est l'équivalent calibré, à taille
appariée. Ces nombres sont un **ordre de grandeur** du coût d'identification de *cette*
sonde sur *ce* protocole, pas une borne sur l'humour lexical.

Deux précisions de lecture. Les puissances sont estimées sur **100 réplications** par case,
soit une précision de **± 8 points** autour de 80 % : les valeurs de « plus petit `d`
détectable » sont des bornes à cette précision, pas des valeurs exactes (à 12 groupes H,
`d = 2.0` atteint 94 % de puissance et `d = 1.5` atteint 79 %). Et la calibration **coûte**
environ une minute de calcul (le temps mesuré est affiché dans la cellule) : c'est le prix
de la mesure.

Ce que la section ajoute au carnet : le verdict `INCONCLUSIVE borne` n'est plus une formule,
c'est une **borne chiffrée**. Et elle désigne le levier : ce n'est pas une feature de plus
qui manque, ce sont des **groupes H** — 12 groupes uniques contre 60 négatifs, c'est le
goulot d'étranglement du dispositif, et il est **borné par le corpus** (14 groupes H au
maximum, 84 unités). La voie reste donc #14033 (substrat taxonomique croissant, > 100 textes
positifs uniques), et le carnet peut désormais dire ce qu'elle rapporte : à 12 groupes H, un
effet de d ≥ 2.0 aurait été vu à 94 % de puissance, et un effet de d = 1 resterait hors de
portée.


In [14]:
# Sortie minimale pour les etages superieurs (gated par #5635 GPU Gate 24 + #14033 substrat taxonomique).
#
# IMPORTANT : ce notebook ne SORT PAS de features pour les etages suivants tel quel -- il
# BORNE un signal lexical sur 6 features deterministes, consommees sur 60 appariements exploratoires
# (12 groupes H uniques x 5 repetitions + 60 negatifs uniques). Les etages superieurs
# (embeddings, modele generatif avec contexte narratif, SAE/J-lens) doivent re-apprendre
# sur des substrats plus expressifs (cf taxonomie sourcee dans le frontmatter : capacites 2 a 6 : lexique baseline -- capacites 2 a 6 = etage 0 couvert ici ; les autres capacites de la taxonomie Attardo/Raskin + Mihalcea/Pulido relevent d etages superieurs).
if verdict == "FEATURE_CANDIDATE":
    print("[P2] Le verdict FEATURE_CANDIDATE : la discrimination des 6 features lexique")
    print("     est exploitable sur ce sous-corpus exploratoire. Une replique avec plus de")
    print("     paires (>100) sur un substrat taxonomique croissant (#14033) reste la voie")
    print("     recommandee pour confirmer. F1 = %.3f" % f1_real)
elif verdict == "SURFACE_SEULE":
    print("[P2] Le verdict SURFACE_SEULE : sur ces 6 features, le strip ne casse pas la")
    print("     discrimination lexicale. Les etages superieurs (embeddings, contexte, SAE)")
    print("     peuvent detecter ce que le lexique rate, mais ne sont pas garantis d aboutir.")
else:
    print("[P2] Le verdict est INCONCLUSIVE borne : les 6 features lexicales deterministes")
    print("     ne discriminent pas au-dela du ratio de classe sur 60 appariements exploratoires.")
    print("     Un etage superieur (embeddings, contexte narratif, SAE) peut seul trancher.")
    print("     Ce notebook borne l etage 0 lexical, ne tranche pas au-dela.")
print()
print("[P2] Etages superieurs gates par #5635 (GPU Gate 24 workspace) et #14033 (substrat")
print("     taxonomique). Aucune action GPU ou embeddings n est prise par ce notebook.")


[P2] Le verdict est INCONCLUSIVE borne : les 6 features lexicales deterministes
     ne discriminent pas au-dela du ratio de classe sur 60 appariements exploratoires.
     Un etage superieur (embeddings, contexte narratif, SAE) peut seul trancher.
     Ce notebook borne l etage 0 lexical, ne tranche pas au-dela.

[P2] Etages superieurs gates par #5635 (GPU Gate 24 workspace) et #14033 (substrat
     taxonomique). Aucune action GPU ou embeddings n est prise par ce notebook.


## Voir aussi

- **#14035** -- issue de reference, protocole falsifiable, acceptance (cible haute).
- **#14032** -- consolidation de la methode de labellisation (GT-28 / GT-28b).
- **#14033** -- substrat taxonomique croissant (corpus apparie + 3 labels + taxonomie sourcee).
- **#26829** -- PR qui ajoute le cadran de puissance (critere D2) a ce carnet.
- **#26911** -- chaine de qualification du corpus : etape 2 (mesure de l'accord de la chaine de labellisation avec une reference humaine, kappa = 0.863), etape 3 (le rejeu de ce carnet a la taille du cadran).
- **`MyIA.AI.Notebooks/GameTheory/GameTheory-18c-Humour-Banc-Python.ipynb`** -- banc humour consolide (#14032) : CORPUS_DUR 120 instances, source de verite.
- **`MyIA.AI.Notebooks/IIT/ICT-Series/ICT-21-SAETrajectories.ipynb`** -- substrat S4 (SAE Qwen3.5-9B), aval + tard.
- **`MyIA.AI.Notebooks/IIT/ICT-Series/ICT-24-WorkspaceIgnition-Python.ipynb`** -- Gate 24, gated etages superieurs (GPU).
- **#5635, #8236, #5105, #4588** -- aval ICT-Series.

- **Note :** ICT-35 -- *HumorCausalProbe-Pilot*, sub-grain exploratoire de #14035, **etage 0 lexical baseline**, GPU-free. Verdict INCONCLUSIVE borne sur 60 appariements exploratoires (12 groupes H uniques x 5 repetitions + 60 negatifs uniques). Mesure StratifiedKFold(n_splits=3) sur 72 unites group-level (unification par moyenne H ; protocole detaille en tete de carnet). L'etape d'apres est #14033 (substrat taxonomique), pas Phase 2 SAE. Si le verdict devient FEATURE_CANDIDATE apres elargissement (>100 textes positifs uniques) ou changement d'etage, ce notebook sera promu en ICT-35 numerote definitif ; sinon il restera un -Pilot archive pour tracabilite.


## À retenir

1. Le banc HLS consolidé (`GameTheory-18c`, #14032) fournit `CORPUS_DUR` = 120 instances annotées en 5 catégories, dont 48 `humour_reussi` (30 blagues manuelles + 18 Argumentum, tous de textes distincts) et 72 non-humour. Le sous-ensemble stratifié de ce carnet en retient **12 groupes H**, réutilisés 5 fois chacun, pour former **60 appariements exploratoires** avec 60 négatifs uniques appariés par longueur (±20 %, quasi tous à delta = 0).
2. Le classifieur lexical group-level atteint `F1 = 0.308 +/- 0.156` sur 72 unités (12 H + 60 N), **à peine au-dessus** de la baseline aléatoire au ratio de classe (`0.286`) : sur ces 6 features, presque rien ne permet de deviner qui est humoristique.
3. Le contrôle de permutation (100 permutations intra-fold de `y_group`, une seule moyenne CV par tirage) donne un null `0.226 +/- 0.092`, IC95 `[0.046, 0.410]` ; 16 moyennes nulles sur 100 atteignent ou dépassent le réel, d'où `p = 0.1683` et un écart `+0.082`, inférieur à un écart-type nul.
4. Le second contrôle (retrait des marqueurs de surface : ponctuation expressive, verbes de recadrage, callbacks) donne `delta_proba = +0.045 +/- 0.172`, IC95 `[-0.169, 0.385]` : l'intervalle traverse 0, et sur 12 H il le fait trivialement — la cellule refuse donc de conclure `SURFACE_SEULE`.
5. Verdict fermé : `INCONCLUSIVE borne`. Ce n'est pas une falsification de l'humour lexical en général, mais la borne la plus précise que ce sous-corpus (72 unités group-level, 12 groupes H, 6 features) puisse porter.
6. Cette borne a une **étendue mesurée** (calibration de la puissance, § Lecture 7) : à 12 groupes H (72 unités), cette sonde détecte à 80 % de puissance un effet de taille `d ≥ 2.0` (puissance mesurée 94 % à `d = 2.0`) ; il en faudrait 48 groupes H (288 unités) pour un effet de `d = 1`, et 24 (144 unités) pour `d = 1.5`. Le F1 observé (0.308) se situe au **72ᵉ centile** du null sans signal (72 tirages nuls sur 100 sont en dessous ou égaux à la valeur observée, 28 la dépassent) : c'est ce que l'instrument produit quand il n'y a rien. Le goulot n'est donc pas le nombre de features, mais le nombre de **groupes H** — d'où la voie #14033 (substrat taxonomique croissant). Les puissances sont estimées sur 100 réplications par case (± 8 points) : ce sont des bornes, pas des valeurs exactes.
7. **Ce que le corpus permet, et pourquoi la taille est plafonnée.** Le rejeu à 12 groupes H (72 unités) est la taille **retenue par ce protocole** : elle est atteignable, mais en deçà du plafond du corpus. La ligne « 24 groupes H / 144 unités » du cadran exigerait 120 négatifs uniques (5 par groupe H) et le corpus n'en compte que 72 ; le **plafond réel** est donc **14 groupes H (84 unités)** — pas 12. L'écart entre la taille demandée par le cadran et la taille retenue est un fait de corpus, pas une négligence de protocole.
8. **La limite de circularité, à ne pas perdre de vue.** Les labels de la classe positive viennent de la chaîne de qualification (#14032 pour la méthode ; #26911 étape 2 pour la **mesure** de l'accord), dont l'accord avec une référence humaine échantillonnée est mesuré dans l'artefact commité (#26924) : **kappa = 0.863** (accord corrigé du hasard ; accord observé 0.917, accord attendu 0.390) et **désaccord brut de 5 items sur 60 (8,3 %)** — sur les cinq classes, pas sur la seule classe positive. Le bruit d'annotation borne donc le signal que cette sonde peut chercher : le F1 se compare au **null intra-fold**, jamais à un oracle humain, et cette borne-là n'est pas levée par l'élargissement à 12 groupes H. Le κ ne donne **aucun plafond de F1** : c'est la matrice de confusion de la Lecture 5 bis qui dit ce que vaut cet accord, classe par classe. Et l'origine des labels est elle-même mesurée : sur les 12 groupes H, **7 sont d'origine heuristique Argumentum** et 5 d'annotation humaine (39 des 60 négatifs sont heuristiques) — le `F1 = 0.308` mesure donc pour l'essentiel une discrimination **interne à l'étiquetage**, pas l'humour au sens humain.

## Vérifiez votre compréhension

1. Pourquoi les 60 appariements ne sont-ils pas 60 paires minimales indépendantes ?
2. Que mesure exactement le contrôle de permutation, et pourquoi ne conserve-t-il qu'une moyenne CV par tirage ?
3. Un `delta_proba` positif suffirait-il à conclure `SURFACE_SEULE` ?
4. Pourquoi le carnet s'arrête-t-il à 12 groupes H, alors que le cadran mentionne jusqu'à 48 ?

<details>
<summary>Réponses</summary>

1. Parce que le sous-ensemble stratifié ne contient que 12 textes `humour_reussi` retenus sur les 48 disponibles : chaque positif est réutilisé 5 fois, sur 5 négatifs distincts. Les 60 appariements ne portent donc que **12 unités positives indépendantes** — le design est exploratoire, et la CV group-level agrège à 72 unités (12 H + 60 N), pas à 60 paires indépendantes.
2. Il permute les étiquettes de `y_group` au niveau global en préservant la composition 12 H / 60 N, refit tout le pipeline à chaque tirage, et ne conserve qu'**une moyenne CV par permutation** : c'est la distribution de ces 100 moyennes qui forme le null, comparée au F1 réel. Une seule moyenne par tirage évite de traiter les folds d'une même permutation comme indépendants.
3. Non. Avec 12 H, `delta_proba = +0.045 +/- 0.172` et un IC95 `[-0.169, 0.385]` qui traverse 0 : sur un effectif aussi petit, un intervalle qui traverse 0 est attendu. `SURFACE_SEULE` exigerait un IC95 entièrement au-dessus de 0 sur un N suffisant ; la cellule déclare donc « pas d'effet clair » plutôt que de trancher.
4. Parce que le protocole consomme **un négatif distinct par appariement** : 5 négatifs par groupe H. Le corpus n'en compte que 72, ce qui plafonne le dispositif à 14 groupes H (84 unités group-level). La ligne « 24 groupes H / 144 unités » du cadran exigerait 120 négatifs uniques : elle est hors d'atteinte ici, et c'est le corpus qui le décide — la voie pour la débloquer est #14033 (substrat taxonomique croissant).

</details>


## Pour aller plus loin

- **Carnet suivant (famille humour)** : [ICT-35b — paires minimales sur substrat SAE](ICT-35b-HumorCausalPairs-SAE-Python.ipynb), tranche 1 de la même étude : la même question, posée sur un design apparié et un autre lecteur que les 6 features lexicales.
- **Carnet suivant déclaré par l'en-tête de ce carnet** : [ICT-36 — F-Lens, mode factored-geometry](ICT-36-FLens-FactoredGeometry-Python.ipynb), grain autonome de la série.
- **Code** : `ict/humor_pairs.py` — corpus, paires et mesure group-level utilisés ici.